In [7]:
import duckdb
import pandas
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

In [ ]:
# Ensure your output directory exists
os.makedirs("../Data/processed_corpus", exist_ok=True)

# Connect to an in-memory DuckDB instance
conn = duckdb.connect()

# Define the path to your JSONL files (you may need to change these)
input_path = "../Data/data_raw/longeval_sci_training_2026_fulltext/documents/*.jsonl"
output_path = "../Data/processed_corpus/longeval_documents.parquet"
# It may be better to keep fullText separate. For now, I combined it with title and abstract.
conn.execute(f"""
    COPY (
        SELECT 
            id::VARCHAR AS doc_id, 
            
            concat_ws('\n\n', title, abstract, fullText) AS original_text,
        
            json_object(
                'arxivId', arxivId,
                'authors', authors,
                'createdDate', createdDate,
                'doi', doi,
                'links', links,
                'magId', magId,
                'oaiIds', oaiIds,
                'publishedDate', publishedDate,
                'pubmedId', pubmedId,
                'updatedDate', updatedDate
            ) AS metadata
            
        FROM read_json_auto('{input_path}')
    ) TO '{output_path}' (FORMAT PARQUET, COMPRESSION ZSTD);
""")

print(f"Successfully created {output_path}")

# Verify the row count
count = conn.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
print(f"Total documents ingested: {count}")

Successfully created ../Data/processed_corpus/longeval_documents.parquet
Total documents ingested: 869902


In [ ]:
# Fetch exactly ONE document's teext
conn = duckdb.connect()
sample_query = "SELECT original_text FROM read_parquet('../Data/processed_corpus/longeval_documents.parquet') LIMIT 1"
sample_text = conn.execute(sample_query).fetchone()[0]

# first paragraph/chunk for a fast test
sample_chunk = sample_text.split('\n\n')[0] 

# Load Gemma locally
model_id = "google/gemma-4-e4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

messages = [
    {"role": "user", "content": f"""You are a strict information extraction system for scientific literature. 
Break the text down into atomic, independent facts. 
Each fact must be fully resolvable on its own without needing the original text.
Output ONLY a valid JSON list of strings. Do not include markdown formatting or conversational filler.

EXAMPLE:
Text: "The RoBERTa baseline achieved an F1 of 0.85. However, our proposed framework improved this to 0.92."
Output: [
  "The RoBERTa baseline achieved an F1 score of 0.85.",
  "The proposed framework achieved an F1 score of 0.92.",
  "The proposed framework improved upon the RoBERTa baseline."
]

YOUR TURN:
Text: {sample_chunk}
Output:"""}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.1)

# Decode
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("RAW MODEL OUTPUT:")
print(response)

Loading weights: 100%|██████████| 2130/2130 [00:00<00:00, 27648.68it/s]


RAW MODEL OUTPUT:
[
  "Two new species were discovered.",
  "The new species belong to the genus Pterostichus.",
  "The new species belong to the subgenus Pseudoferonina.",
  "The species are part of the family Carabidae.",
  "The species are part of the tribe Pterostichini.",
  "The species were found in the mountains of central Idaho, U.S.A."
]


In [11]:
model_id = "cross-encoder/nli-deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

# Map the labels to avoid index mismatch
label_map = model.config.label2id
entail_idx = label_map.get('entailment', label_map.get('LABEL_0')) 
contra_idx = label_map.get('contradiction', label_map.get('LABEL_2'))

query_nuggets = [
    "New species were discovered.",
    "The species belong to the family Carabidae.",
    "The species were discovered in the United States."
]

document_facts = [
    "Two new species were discovered.",
    "The new species belong to the genus Pterostichus.",
    "The new species belong to the subgenus Pseudoferonina.",
    "The species are part of the family Carabidae.",
    "The species are part of the tribe Pterostichini.",
    "The species were found in the mountains of central Idaho, U.S.A."
]

print("QUERY NUGGETS:")
for q in query_nuggets:
    print(f"- {q}")
print("-" * 50)

scored_facts = []

for d_fact in document_facts:
    best_score = 0
    best_signal = "NEUTRAL"
    best_match = ""
    best_entail = 0.0
    best_contra = 0.0
    
    # Check this document fact against EVERY query nugget
    for q_nugget in query_nuggets:
        inputs = tokenizer(d_fact, q_nugget, return_tensors="pt", truncation=True).to(device)
        
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=1)[0]
            
        p_entail = probs[entail_idx].item()
        p_contra = probs[contra_idx].item()
        
        # Track the strongest logical connection this doc fact has to ANY query fact
        local_max = max(p_entail, p_contra)
        if local_max > best_score:
            best_score = local_max
            best_signal = "ENTAILMENT" if p_entail > p_contra else "CONTRADICTION"
            best_match = q_nugget
            best_entail = p_entail
            best_contra = p_contra
            
    scored_facts.append({
        "fact": d_fact,
        "score": best_score,
        "signal": best_signal,
        "matched_query": best_match,
        "entailment": best_entail,
        "contradiction": best_contra
    })

# Sort by the highest logical signal
scored_facts.sort(key=lambda x: x["score"], reverse=True)

# Display the results
for item in scored_facts:
    # Force low-confidence scores to display as NEUTRAL
    display_signal = item["signal"]
    if item["score"] < 0.5:
        display_signal = "NEUTRAL"
        
    print(f"Fact: {item['fact']}")
    print(f"Strongest Match: '{item['matched_query']}'")
    print(f"Signal: {display_signal} (Score: {item['score']:.4f})")
    print(f"  [Entail: {item['entailment']:.4f} | Contra: {item['contradiction']:.4f}]")
    print("-" * 50)

# Take the Top 5 most logically relevant facts for the generator
top_facts = [item["fact"] for item in scored_facts[:5]]

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 24025.22it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


QUERY NUGGETS:
- New species were discovered.
- The species belong to the family Carabidae.
- The species were discovered in the United States.
--------------------------------------------------
Fact: The species are part of the family Carabidae.
Strongest Match: 'The species belong to the family Carabidae.'
Signal: ENTAILMENT (Score: 0.9978)
  [Entail: 0.9978 | Contra: 0.0001]
--------------------------------------------------
Fact: Two new species were discovered.
Strongest Match: 'New species were discovered.'
Signal: ENTAILMENT (Score: 0.9961)
  [Entail: 0.9961 | Contra: 0.0001]
--------------------------------------------------
Fact: The species were found in the mountains of central Idaho, U.S.A.
Strongest Match: 'The species were discovered in the United States.'
Signal: ENTAILMENT (Score: 0.9957)
  [Entail: 0.9957 | Contra: 0.0001]
--------------------------------------------------
Fact: The new species belong to the genus Pterostichus.
Strongest Match: 'New species were discov

In [14]:
gemma_id = "google/gemma-4-e4b-it"
gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_id)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
gemma_model = AutoModelForCausalLM.from_pretrained(gemma_id).to(device)


top_facts_text = "\n".join(f"- {fact}" for fact in top_facts)
user_query = "Were any new Carabidae found in the United States?"

# Hardcode the exact Gemma instruction tokens
prompt = f"""<start_of_turn>user
You are an expert scientific evaluator. Answer the User Query using ONLY the provided Facts. 

RULES:
1. If the facts provide the answer, synthesize them into a clear, concise sentence.
2. If the facts contradict each other, explicitly state the contradiction.
3. If the facts do not contain enough information to answer the query, output EXACTLY: "Insufficient evidence."

EXAMPLE 1 (Synthesis):
Query: Did the proposed model beat the baseline?
Facts: 
- The RoBERTa baseline achieved an F1 of 0.85.
- The proposed LogicRAG framework achieved an F1 of 0.92.
Output: Yes, the proposed LogicRAG framework (F1 0.92) outperformed the RoBERTa baseline (F1 0.85).

EXAMPLE 2 (Contradiction):
Query: What was the primary cause of the error?
Facts:
- Smith et al. (2023) states the error was caused by sensor drift.
- Jones (2024) claims the error was due to sub-pixel heterogeneity.
Output: The provided facts contradict each other; Smith et al. attribute the error to sensor drift, while Jones attributes it to sub-pixel heterogeneity.

EXAMPLE 3 (Missing Evidence):
Query: What was the learning rate used during training?
Facts:
- The model was trained for 50 epochs.
- An Adam optimizer was used.
Output: Insufficient evidence.

YOUR TURN:
Query: {user_query}
Facts:
{top_facts_text}
Output:<end_of_turn>
<start_of_turn>model
"""

# Tokenize the raw string
inputs = gemma_tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
outputs = gemma_model.generate(**inputs, max_new_tokens=512, temperature=0.2)
final_answer = gemma_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("FINAL RAG ANSWER:")
print(final_answer.strip())

Loading weights: 100%|██████████| 2130/2130 [00:00<00:00, 30421.55it/s]


FINAL RAG ANSWER:
Yes, two new species belonging to the genus Pterostichus were discovered in the mountains of central Idaho, U.S.A., and they are part of the family Carabidae and tribe Pterostichini.
